# Exploring pubtator data for parser

In [1]:
## CX: allows multiple lines of code to print from one code block
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Trying to stop ruff check from using this file
# ruff: noqa

## put in parser (format)
import pathlib
import polars as pl
import html

In [2]:
## useful function: polars version
def check_if_contains(df, column_name, patterns):
    for i in patterns:
        temp = df.filter(pl.col(column_name).str.contains(i, literal=True))
        if temp.shape[0] > 0:
            print(f'{i} count: {temp.shape[0]}')
            
## using literal=True seems to cover special char
delimiters = [",", ";", "-", "_", "|", " "]

<div class="alert alert-block alert-danger">
    
This notebook was originally written using the `2026-03-17` data files from the [FTP site](https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/)

## relation2pubtator3

### Loading file

In [3]:
## using downloaded files 
## paths to raw data files

base_file_path = pathlib.Path.home().joinpath("Desktop", "pubtator_files", "2026_03_17")

relation_path = base_file_path.joinpath("relation2pubtator3.gz")

In [4]:
SOURCE_COLUMNS = ["pmid", "relation", "entity1", "entity2"]

In [5]:
relation_df = pl.read_csv(relation_path,
                          separator="\t",
                          has_header=False,
                          new_columns=SOURCE_COLUMNS,
                         )

In [6]:
relation_df.shape

(39163686, 4)

In [7]:
relation_df.head()

pmid,relation,entity1,entity2
i64,str,str,str
35378878,"""associate""","""Chemical|MESH:D003911""","""Disease|MESH:D005334"""
35378880,"""treat""","""Chemical|MESH:D010866""","""Disease|MESH:D060585"""
35378880,"""associate""","""Chemical|MESH:D010866""","""Chemical|MESH:D011108"""
35378880,"""treat""","""Chemical|MESH:D010866""","""Disease|MESH:D009181"""
35378880,"""treat""","""Chemical|MESH:D011108""","""Disease|MESH:D060585"""


### pmid column

i64 (int64) means these values are single IDs w/o prefix. 

In [8]:
## add PMID prefixes
##   do early - after merging rows/making set, annoying to add prefix

relation_df = \
relation_df.with_columns(
    pl.concat_str([pl.lit("PMID:"), pl.col("pmid").cast(pl.Utf8)]).alias("pmid"))

relation_df.head()

pmid,relation,entity1,entity2
str,str,str,str
"""PMID:35378878""","""associate""","""Chemical|MESH:D003911""","""Disease|MESH:D005334"""
"""PMID:35378880""","""treat""","""Chemical|MESH:D010866""","""Disease|MESH:D060585"""
"""PMID:35378880""","""associate""","""Chemical|MESH:D010866""","""Chemical|MESH:D011108"""
"""PMID:35378880""","""treat""","""Chemical|MESH:D010866""","""Disease|MESH:D009181"""
"""PMID:35378880""","""treat""","""Chemical|MESH:D011108""","""Disease|MESH:D060585"""


### entity cols: initial processing

Split original "entity" cols into Type + ID

In [9]:
## split entity1/entity2 cols by | into type + ID

DESIRED_COLS = [
    "pmid",
    "relation",
    "entity1_type",
    "entity1_id",
    "entity2_type",
    "entity2_id",
]

relation_df = relation_df \
    .with_columns(
        pl.col("entity1").str.split_exact("|", 1).alias("entity1_parts"),
        pl.col("entity2").str.split_exact("|", 1).alias("entity2_parts"),
    ) \
    .with_columns(
        pl.col("entity1_parts").struct.field("field_0").alias("entity1_type"),
        pl.col("entity1_parts").struct.field("field_1").alias("entity1_id"),
        pl.col("entity2_parts").struct.field("field_0").alias("entity2_type"),
        pl.col("entity2_parts").struct.field("field_1").alias("entity2_id"),
    ) \
    .select(DESIRED_COLS)

In [10]:
relation_df.head()

pmid,relation,entity1_type,entity1_id,entity2_type,entity2_id
str,str,str,str,str,str
"""PMID:35378878""","""associate""","""Chemical""","""MESH:D003911""","""Disease""","""MESH:D005334"""
"""PMID:35378880""","""treat""","""Chemical""","""MESH:D010866""","""Disease""","""MESH:D060585"""
"""PMID:35378880""","""associate""","""Chemical""","""MESH:D010866""","""Chemical""","""MESH:D011108"""
"""PMID:35378880""","""treat""","""Chemical""","""MESH:D010866""","""Disease""","""MESH:D009181"""
"""PMID:35378880""","""treat""","""Chemical""","""MESH:D011108""","""Disease""","""MESH:D060585"""


In [82]:
## no null values!

relation_df.null_count()

pmid,relation,entity1_type,entity1_id,entity2_type,entity2_id
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


In [11]:
## what types are there

relation_df["entity1_type"].value_counts().sort("count", descending=True).show(limit=None)

relation_df["entity2_type"].value_counts().sort("count", descending=True).show(limit=None)

entity1_type,count
str,u32
"""Chemical""",24798573
"""Disease""",10305438
"""Gene""",3765823
"""DNAMutation""",277527
"""ProteinMutation""",13583
"""SNP""",2736
"""Mutation""",6


entity2_type,count
str,u32
"""Gene""",18805153
"""Disease""",12734764
"""Chemical""",6795242
"""ProteinMutation""",566827
"""SNP""",217085
"""DNAMutation""",44069
"""Mutation""",546


Same entity types in entity1 and 2

Next, check the ID format for each type: complex structure? prefixes?

### Current main types/IDs

**Chemical: MESH** - no transforms needed  
Should be single values (no delimiters). Has Translator-standard prefix.


**Disease: MESH, OMIM** - no transforms needed  
Should be single values (no delimiters). Has Translator-standard prefix.

**Gene: no prefix in data, should be NCBIGene** based on paper [normalization section](https://academic.oup.com/nar/article/52/W1/W540/7640526#469437536) and supplementary table 7.  
Should be single values (no delimiters).

In [12]:
delimiters

[',', ';', '-', '_', '|', ' ']

#### Chemical

In [13]:
## look for delimiters - none, single values

print("Entity1\n")

check_if_contains(relation_df.filter((pl.col("entity1_type") == "Chemical")), 
                  "entity1_id", 
                  delimiters)

print("Entity2\n")

check_if_contains(relation_df.filter((pl.col("entity2_type") == "Chemical")), 
                  "entity2_id", 
                  delimiters)

Entity1

Entity2



In [14]:
## look for prefixes that aren't "MESH:" - none

relation_df.filter((pl.col("entity1_type") == "Chemical") &
                   (pl.col("entity1_id").str.starts_with("MESH:").not_()))

relation_df.filter((pl.col("entity2_type") == "Chemical") &
                   (pl.col("entity2_id").str.starts_with("MESH:").not_()))

pmid,relation,entity1_type,entity1_id,entity2_type,entity2_id
str,str,str,str,str,str


pmid,relation,entity1_type,entity1_id,entity2_type,entity2_id
str,str,str,str,str,str


#### Disease

In [15]:
## look for delimiters - none, single values

print("Entity1\n")

check_if_contains(relation_df.filter((pl.col("entity1_type") == "Disease")), 
                  "entity1_id", 
                  delimiters)

print("Entity2\n")

check_if_contains(relation_df.filter((pl.col("entity2_type") == "Disease")), 
                  "entity2_id", 
                  delimiters)

Entity1

Entity2



In [16]:
## look for other prefixes - none

relation_df.filter((pl.col("entity1_type") == "Disease") &
                   (pl.col("entity1_id").str.starts_with("MESH:").not_()) &
                   (pl.col("entity1_id").str.starts_with("OMIM:").not_()))

relation_df.filter((pl.col("entity2_type") == "Disease") &
                   (pl.col("entity2_id").str.starts_with("MESH:").not_()) &
                   (pl.col("entity2_id").str.starts_with("OMIM:").not_()))

pmid,relation,entity1_type,entity1_id,entity2_type,entity2_id
str,str,str,str,str,str


pmid,relation,entity1_type,entity1_id,entity2_type,entity2_id
str,str,str,str,str,str


#### Gene

In [17]:
## look for delimiters - none, single values

print("Entity1\n")

check_if_contains(relation_df.filter((pl.col("entity1_type") == "Gene")), 
                  "entity1_id", 
                  delimiters)

print("Entity2\n")

check_if_contains(relation_df.filter((pl.col("entity2_type") == "Gene")), 
                  "entity2_id", 
                  delimiters)

Entity1

Entity2



In [18]:
## Gene: no prefix (:)

relation_df.filter((pl.col("entity1_type") == "Gene") &
                   (pl.col("entity1_id").str.contains(":", literal=True)))

relation_df.filter((pl.col("entity2_type") == "Gene") &
                   (pl.col("entity2_id").str.contains(":", literal=True)))

pmid,relation,entity1_type,entity1_id,entity2_type,entity2_id
str,str,str,str,str,str


pmid,relation,entity1_type,entity1_id,entity2_type,entity2_id
str,str,str,str,str,str


### Variant types/IDs

[2026-03]

Currently, we don't plan to ingest rows with these types because NodeNorm doesn't include any variant namespaces:
* "DNAMutation"
* "ProteinMutation"
* "SNP"
* "Mutation"

These types seem to fall under what the [paper](https://academic.oup.com/nar/article/52/W1/W540/7640526) calls "genetic variants" and the [FTP site](https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/README.txt) calls "mutations". 

#### DNAMutation

The ID values are complex, dict-like.

In [19]:
## notice issue

relation_df.filter((pl.col("entity1_type") == "DNAMutation")).head()

pmid,relation,entity1_type,entity1_id,entity2_type,entity2_id
str,str,str,str,str,str
"""PMID:35378956""","""cause""","""DNAMutation""","""RS#:568970047;HGVS:c.464G>A;Co…","""Disease""","""MESH:C579932"""
"""PMID:35379107""","""associate""","""DNAMutation""","""RS#:67376798;HGVS:c.2846A>T;Co…","""Disease""","""MESH:D005770"""
"""PMID:35379107""","""associate""","""DNAMutation""","""RS#:55886062;HGVS:c.1679T>G;Co…","""Disease""","""MESH:D005770"""
"""PMID:35379107""","""associate""","""DNAMutation""","""RS#:3918290;HGVS:c.1905+1G>A;C…","""Disease""","""MESH:D005770"""
"""PMID:35379107""","""associate""","""DNAMutation""","""RS#:75017182;HGVS:c.1129_5923C…","""Disease""","""MESH:D005770"""


In [20]:
## look at rows with this type

relation_DNAMutation_1 = relation_df.filter((pl.col("entity1_type") == "DNAMutation")) 
relation_DNAMutation_1.shape[0]

relation_DNAMutation_2 = relation_df.filter((pl.col("entity2_type") == "DNAMutation")) 
relation_DNAMutation_2.shape[0]

277527

44069

In [21]:
## look at full values

with pl.Config(fmt_str_lengths=100):
    relation_DNAMutation_1["entity1_id"].head()
    
    relation_DNAMutation_2["entity2_id"].head()

entity1_id
str
"""RS#:568970047;HGVS:c.464G>A;CorrespondingGene:57010"""
"""RS#:67376798;HGVS:c.2846A>T;CorrespondingGene:1806"""
"""RS#:55886062;HGVS:c.1679T>G;CorrespondingGene:1806"""
"""RS#:3918290;HGVS:c.1905+1G>A;CorrespondingGene:1806"""
"""RS#:75017182;HGVS:c.1129_5923C>G;CorrespondingGene:1806"""
"""RS#:1232331611;HGVS:c.3G>A;CorrespondingGene:58497"""
"""RS#:1232331611;HGVS:c.3G>A;CorrespondingGene:58497"""
"""RS#:1232331611;HGVS:c.3G>A;CorrespondingGene:58497"""
"""RS#:1232331611;HGVS:c.3G>A;CorrespondingGene:58497"""


entity2_id
str
"""HGVS:c.2A>A"""
"""HGVS:c.2063A>G"""
"""HGVS:c.10C>C"""
"""HGVS:c.64G>T"""
"""HGVS:p.G64S"""
"""RS#:1364963022;HGVS:c.273G>A;CorrespondingGene:2099"""
"""HGVS:c.143G>A"""
"""RS#:747843990;HGVS:c.327G>C;CorrespondingGene:3242"""
"""HGVS:c.2A>A;CorrespondingGene:135"""


Some values include [html character entities](https://www.w3schools.com/HTML/html_entities.asp) that start with `&` and **end with `;`**. But `;` is also used as a delimiter between diff key-value pairs.  
**So, transform them into regular char first.**

(Context: noticed when trying to split on `;` and turn into dict -> diff between len and keys) 

In [24]:
check_if_contains(relation_DNAMutation_1, "entity1_id", [r"&"])
check_if_contains(relation_DNAMutation_2, "entity2_id", [r"&"])

## look at full values
with pl.Config(fmt_str_lengths=100):
    relation_DNAMutation_1.filter((pl.col("entity1_id").str.contains(r"&")))["entity1_id"].head()
    
    relation_DNAMutation_2.filter((pl.col("entity2_id").str.contains(r"&")))["entity2_id"].head()

& count: 37
& count: 6746


entity1_id
str
"""HGVS:c.&lt;IVS2NT+1G>T;CorrespondingGene:6928"""
"""HGVS:c.140&lt;=5G>A;CorrespondingGene:1497"""
"""HGVS:c.1A&gt;C;CorrespondingGene:1401"""
"""HGVS:c.1385A&gt;G"""
"""RS#:1449772752;HGVS:c.1157T&gt;C;CorrespondingGene:462"""
"""HGVS:c.2A&gt;A"""
"""HGVS:c.198A&gt;G"""
"""HGVS:c.198A&gt;G"""
"""HGVS:c.206C&gt;T;CorrespondingGene:9935"""


entity2_id
str
"""RS#:1447681977;HGVS:c.6464T&gt;C;CorrespondingGene:55714"""
"""HGVS:c.941C&gt;T;CorrespondingGene:55714"""
"""HGVS:c.34311A&gt;C;CorrespondingGene:672"""
"""HGVS:c.2A&gt;A;CorrespondingGene:135"""
"""HGVS:c.2A&gt;A;CorrespondingGene:135"""
"""HGVS:c.2A&gt;A;CorrespondingGene:135"""
"""HGVS:c.2A&gt;A;CorrespondingGene:135"""
"""HGVS:c.940G&gt;T;CorrespondingGene:2070"""
"""HGVS:c.548C&gt;G;CorrespondingGene:2070"""


In [25]:
## using html package on a string

html.unescape("RS#:1447681977;HGVS:c.6464T&gt;C;CorrespondingGene:55714")

'RS#:1447681977;HGVS:c.6464T>C;CorrespondingGene:55714'

**NOTE**: Should be okay to do this col-wise in whole df: shouldn't harm values

In [26]:
## col-wise: turn these into regular strings with html package

relation_DNAMutation_1 = relation_DNAMutation_1.with_columns(
    pl.col("entity1_id")
    .map_elements(html.unescape, return_dtype=pl.Utf8)
    .alias("entity1_id")
)

relation_DNAMutation_2 = relation_DNAMutation_2.with_columns(
    pl.col("entity2_id")
    .map_elements(html.unescape, return_dtype=pl.Utf8)
    .alias("entity2_id")
)

In [28]:
## check: if nothing prints, html character entities are gone

check_if_contains(relation_DNAMutation_1, "entity1_id", [r"&"])
check_if_contains(relation_DNAMutation_2, "entity2_id", [r"&"])

In [30]:
## check for other html strings. url-encoded: %20, tags: </b>
## if nothing prints, nothing found

other_html = [r"%", r"</"]

check_if_contains(relation_DNAMutation_1, "entity1_id", other_html)
check_if_contains(relation_DNAMutation_2, "entity2_id", other_html)

EDA: Inspecting the possible keys by turning values into dicts. 

In [28]:
dnamutation_keys = set()
longest = 0

## entity 1
for s in relation_DNAMutation_1["entity1_id"]:
    n_items = len(s.split(";"))
    if n_items > longest:
        longest = n_items
    if n_items > 3:
        print(s)
    
    temp = dict(item.split(":", 1) for item in s.split(";") if ":" in item)
    
    dnamutation_keys.update(temp.keys())

In [29]:
dnamutation_keys

longest

temp

{'CorrespondingGene', 'HGVS', 'RS#'}

3

{'HGVS': 'c.3A>T'}

In [30]:
## entity 2
for s in relation_DNAMutation_2["entity2_id"]:
    n_items = len(s.split(";"))
    if n_items > longest:
        longest = n_items
    if n_items > 3:
        print(s)
    
    temp = dict(item.split(":", 1) for item in s.split(";") if ":" in item)
    
    dnamutation_keys.update(temp.keys())

In [31]:
dnamutation_keys

longest

temp

{'CorrespondingGene', 'HGVS', 'RS#'}

3

{'RS#': '1801133', 'HGVS': 'c.677C>T', 'CorrespondingGene': '4524'}

[2026-03]

So there's 3 possible keys:
* `RS#`: dbSNP ID? If so, [missing 'rs' in value](https://bioregistry.io/registry/dbsnp)
* `CorrespondingGene`: no prefix in data, should be NCBIGene ID
* `HGVS`: [nomenclature for variant description](https://hgvs-nomenclature.org/stable/), not regular database ID. And **these are NOT complete, standard values** - [missing sequence and transcript ID](https://hgvs-nomenclature.org/stable/recommendations/DNA/substitution/)

**NOTE**: The whole ID string can be used in group-by/merging rows. Then this dict parsing can be done in `transform_record` step.

Easier to code, preprocessing doesn't depend on knowing keys, less iteration over whole df.

Reviewed possible multi-value delimiters: based on a quick skim, all values found were part of `HGVS` values. 

In [31]:
delimiters

[',', ';', '-', '_', '|', ' ']

In [32]:
## do a delimiter check on a more limited set
## if nothing prints, nothing found

var_delimiters = [",", "-", "_", "|", " "]

check_if_contains(relation_DNAMutation_1, "entity1_id", var_delimiters)
check_if_contains(relation_DNAMutation_2, "entity2_id", var_delimiters)

, count: 3271
- count: 39326
_ count: 17287
, count: 807
- count: 4583
_ count: 1069


In [37]:
## look at full values
with pl.Config(fmt_str_lengths=100):
    relation_DNAMutation_1.filter((pl.col("entity1_id").str.contains("_")))["entity1_id"].head()

entity1_id
str
"""RS#:75017182;HGVS:c.1129_5923C>G;CorrespondingGene:1806"""
"""HGVS:g.25679_25679delA;CorrespondingGene:1727"""
"""HGVS:c.824_825insC;CorrespondingGene:1727"""
"""HGVS:c.824_825insC;CorrespondingGene:1727"""
"""HGVS:g.25679_25679delA;CorrespondingGene:1727"""
"""HGVS:c.824_825insC;CorrespondingGene:1727"""
"""HGVS:g.25679_25679delA;CorrespondingGene:1727"""
"""HGVS:c.317_320dupAGTG;CorrespondingGene:1161"""
"""HGVS:c.343_344delCT;CorrespondingGene:268"""


#### ProteinMutation - NOT DONE

The ID values are complex, dict-like (look like DNAMutation's). 

In [43]:
## quick look

relation_df.filter((pl.col("entity1_type") == "ProteinMutation")).head()

pmid,relation,entity1_type,entity1_id,entity2_type,entity2_id
str,str,str,str,str,str
"""PMID:35378893""","""associate""","""ProteinMutation""","""HGVS:p.G323D;CorrespondingGene…","""ProteinMutation""","""HGVS:p.Y137H;CorrespondingGene…"
"""PMID:34903070""","""associate""","""ProteinMutation""","""RS#:12934922;HGVS:p.R267S;Corr…","""ProteinMutation""","""RS#:7501331;HGVS:p.A379V;Corre…"
"""PMID:34050359""","""associate""","""ProteinMutation""","""RS#:113488022;HGVS:p.V600E;Cor…","""ProteinMutation""","""RS#:760043106;HGVS:p.I195N;Cor…"
"""PMID:34121011""","""associate""","""ProteinMutation""","""HGVS:p.K4326E;CorrespondingGen…","""ProteinMutation""","""HGVS:p.L1412KfsX16;Correspondi…"
"""PMID:29625985""","""associate""","""ProteinMutation""","""HGVS:p.H336R;CorrespondingGene…","""ProteinMutation""","""HGVS:p.L541P;CorrespondingGene…"


#### SNP - NOT DONE

The ID values are complex, dict-like (look like DNAMutation's, maybe w/o HGVS?). 

In [44]:
## quick look

relation_df.filter((pl.col("entity1_type") == "SNP")).head()

pmid,relation,entity1_type,entity1_id,entity2_type,entity2_id
str,str,str,str,str,str
"""PMID:21127300""","""associate""","""SNP""","""RS#:10455872;CorrespondingGene…","""SNP""","""RS#:3123629;CorrespondingGene:…"
"""PMID:35998284""","""associate""","""SNP""","""RS#:2241766;CorrespondingGene:…","""SNP""","""RS#:3774261;CorrespondingGene:…"
"""PMID:35617280""","""associate""","""SNP""","""RS#:9527025;CorrespondingGene:…","""SNP""","""RS#:9536314;CorrespondingGene:…"
"""PMID:27951722""","""associate""","""SNP""","""RS#:2020903;CorrespondingGene:…","""SNP""","""RS#:4646034;CorrespondingGene:…"
"""PMID:36723865""","""associate""","""SNP""","""RS#:2812377;CorrespondingGene:…","""SNP""","""RS#:3136658;CorrespondingGene:…"


#### Mutation - NOT DONE

Looks like a variant description, not an ID

In [45]:
## quick look

relation_df.filter((pl.col("entity1_type") == "Mutation")).head()

pmid,relation,entity1_type,entity1_id,entity2_type,entity2_id
str,str,str,str,str,str
"""PMID:32468074""","""associate""","""Mutation""","""Chr8:18765448-18804898del""","""ProteinMutation""","""RS#:113488022;HGVS:p.V600E;Cor…"
"""PMID:32468074""","""associate""","""Mutation""","""Chr5:161330882-161336769del""","""ProteinMutation""","""RS#:113488022;HGVS:p.V600E;Cor…"
"""PMID:32468074""","""associate""","""Mutation""","""Chr9:22046750-22097364dup""","""ProteinMutation""","""RS#:113488022;HGVS:p.V600E;Cor…"
"""PMID:32468074""","""associate""","""Mutation""","""Chr5:161330882-161336769del""","""ProteinMutation""","""RS#:113488022;HGVS:p.V600E;Cor…"
"""PMID:32468074""","""associate""","""Mutation""","""Chr8:18765448-18804898del""","""ProteinMutation""","""RS#:113488022;HGVS:p.V600E;Cor…"


### relation/combos

In [38]:
relation_df["relation"].value_counts().sort("count", descending=True).show(limit=None)

relation,count
str,u32
"""associate""",18173612
"""treat""",7149232
"""negative_correlate""",4006623
"""cause""",3677170
"""positive_correlate""",3394019
"""stimulate""",770147
"""inhibit""",609787
"""cotreat""",552752
"""compare""",530067


In [39]:
relation_df

pmid,relation,entity1_type,entity1_id,entity2_type,entity2_id
str,str,str,str,str,str
"""PMID:35378878""","""associate""","""Chemical""","""MESH:D003911""","""Disease""","""MESH:D005334"""
"""PMID:35378880""","""treat""","""Chemical""","""MESH:D010866""","""Disease""","""MESH:D060585"""
"""PMID:35378880""","""associate""","""Chemical""","""MESH:D010866""","""Chemical""","""MESH:D011108"""
"""PMID:35378880""","""treat""","""Chemical""","""MESH:D010866""","""Disease""","""MESH:D009181"""
"""PMID:35378880""","""treat""","""Chemical""","""MESH:D011108""","""Disease""","""MESH:D060585"""
…,…,…,…,…,…
"""PMID:41822489""","""treat""","""Chemical""","""MESH:D000077150""","""Disease""","""MESH:D013274"""
"""PMID:41822489""","""cotreat""","""Chemical""","""MESH:D000077150""","""Chemical""","""MESH:D002955"""
"""PMID:41822489""","""negative_correlate""","""Chemical""","""MESH:D002955""","""Gene""","""9825"""


In [50]:
triple_cols = ["entity1_type", "relation", "entity2_type"]

combo_counts = (
    relation_df.group_by(triple_cols)
      .len()
)

In [51]:
combo_counts.sort(triple_cols).show(limit=None)

entity1_type,relation,entity2_type,len
str,str,str,u32
"""Chemical""","""associate""","""Chemical""",3012947
"""Chemical""","""associate""","""DNAMutation""",15682
"""Chemical""","""associate""","""Disease""",2125918
"""Chemical""","""associate""","""Gene""",2116211
"""Chemical""","""associate""","""ProteinMutation""",60623
"""Chemical""","""associate""","""SNP""",11069
"""Chemical""","""cause""","""Disease""",3198130
"""Chemical""","""compare""","""Chemical""",530067
"""Chemical""","""cotreat""","""Chemical""",552752


### Filter out variant types

In [53]:
EXCLUDED_ENTITY_TYPES = ["DNAMutation", "ProteinMutation", "SNP", "Mutation"]

In [54]:
relation_df_filtered = relation_df.filter(
            ~(
                pl.col("entity1_type").is_in(EXCLUDED_ENTITY_TYPES)
                | pl.col("entity2_type").is_in(EXCLUDED_ENTITY_TYPES)
            )
        )

In [56]:
relation_df_filtered.shape[0]

relation_df_filtered.shape[0] / relation_df.shape[0]

38073675

0.9721678138263083

In [57]:
combo_counts_filtered = (
    relation_df_filtered.group_by(triple_cols)
      .len()
)

In [59]:
combo_counts_filtered.sort(["relation", "entity1_type", "entity2_type"]) \
    .select(["entity1_type", "relation", "entity2_type", "len"]) \
    .show(limit=None)

entity1_type,relation,entity2_type,len
str,str,str,u32
"""Chemical""","""associate""","""Chemical""",3012947
"""Chemical""","""associate""","""Disease""",2125918
"""Chemical""","""associate""","""Gene""",2116211
"""Disease""","""associate""","""Gene""",8348035
"""Gene""","""associate""","""Gene""",2059527
"""Chemical""","""cause""","""Disease""",3198130
"""Chemical""","""compare""","""Chemical""",530067
"""Chemical""","""cotreat""","""Chemical""",552752
"""Chemical""","""drug_interact""","""Chemical""",7523


In [68]:
combo_counts_filtered["relation"].value_counts().sort("relation").show(limit=None)

relation,count
str,u32
"""associate""",5
"""cause""",1
"""compare""",1
"""cotreat""",1
"""drug_interact""",1
"""inhibit""",1
"""interact""",3
"""negative_correlate""",3
"""positive_correlate""",3


### EDA: Merge by ID 1, relation, ID 2 combos

In [70]:
GROUP_BY_COLUMNS = ["entity1_id", "relation", "entity2_id"]
SET_COLUMNS = ["pmid", "entity1_type", "entity2_type"]

In [74]:
merge1 = (relation_df_filtered
        .group_by(GROUP_BY_COLUMNS)
        .agg([
            pl.col(column).drop_nulls().unique().sort().alias(f"{column}_set")
            for column in SET_COLUMNS
        ]))

In [75]:
merge1

entity1_id,relation,entity2_id,pmid_set,entity1_type_set,entity2_type_set
str,str,str,list[str],list[str],list[str]
"""10468""","""associate""","""3161""","[""PMID:41271182""]","[""Gene""]","[""Gene""]"
"""MESH:D002186""","""associate""","""MESH:D019840""","[""PMID:35450980""]","[""Chemical""]","[""Chemical""]"
"""MESH:C585131""","""treat""","""MESH:D010146""","[""PMID:38484627""]","[""Chemical""]","[""Disease""]"
"""MESH:C000626766""","""associate""","""6241""","[""PMID:30027502""]","[""Chemical""]","[""Gene""]"
"""MESH:C112381""","""negative_correlate""","""MESH:D016572""","[""PMID:2549977"", ""PMID:27313091"", ""PMID:7683296""]","[""Chemical""]","[""Chemical""]"
…,…,…,…,…,…
"""MESH:D014867""","""associate""","""92745""","[""PMID:2306457""]","[""Chemical""]","[""Gene""]"
"""MESH:C039602""","""associate""","""10891""","[""PMID:38749332""]","[""Chemical""]","[""Gene""]"
"""MESH:D010869""","""positive_correlate""","""25645""","[""PMID:12522078"", ""PMID:19429833""]","[""Chemical""]","[""Gene""]"


In [84]:
merge1.filter(pl.col("entity1_type_set").list.len() > 1)

entity1_id,relation,entity2_id,pmid_set,entity1_type_set,entity2_type_set
str,str,str,list[str],list[str],list[str]
"""MESH:C562477""","""associate""","""1571""","[""PMID:10851284"", ""PMID:15456790"", … ""PMID:9877205""]","[""Chemical"", ""Disease""]","[""Gene""]"
"""MESH:C536778""","""associate""","""7124""","[""PMID:18462845"", ""PMID:41640562""]","[""Chemical"", ""Disease""]","[""Gene""]"
"""MESH:C564403""","""associate""","""56997""","[""PMID:18319072"", ""PMID:24048965"", … ""PMID:41769026""]","[""Chemical"", ""Disease""]","[""Gene""]"
"""MESH:C564403""","""associate""","""2110""","[""PMID:17412732"", ""PMID:18836889"", … ""PMID:23727839""]","[""Chemical"", ""Disease""]","[""Gene""]"


In [85]:
merge1.filter(pl.col("entity2_type_set").list.len() > 1)

entity1_id,relation,entity2_id,pmid_set,entity1_type_set,entity2_type_set
str,str,str,list[str],list[str],list[str]
"""MESH:C024989""","""associate""","""MESH:C564403""","[""PMID:17442627"", ""PMID:18387363"", … ""PMID:9266503""]","[""Chemical""]","[""Chemical"", ""Disease""]"


In [89]:
multi_types = merge1.filter((pl.col("entity1_type_set").list.len() > 1) |
                            (pl.col("entity2_type_set").list.len() > 1))

multi_types

entity1_id,relation,entity2_id,pmid_set,entity1_type_set,entity2_type_set
str,str,str,list[str],list[str],list[str]
"""MESH:C562477""","""associate""","""1571""","[""PMID:10851284"", ""PMID:15456790"", … ""PMID:9877205""]","[""Chemical"", ""Disease""]","[""Gene""]"
"""MESH:C536778""","""associate""","""7124""","[""PMID:18462845"", ""PMID:41640562""]","[""Chemical"", ""Disease""]","[""Gene""]"
"""MESH:C024989""","""associate""","""MESH:C564403""","[""PMID:17442627"", ""PMID:18387363"", … ""PMID:9266503""]","[""Chemical""]","[""Chemical"", ""Disease""]"
"""MESH:C564403""","""associate""","""56997""","[""PMID:18319072"", ""PMID:24048965"", … ""PMID:41769026""]","[""Chemical"", ""Disease""]","[""Gene""]"
"""MESH:C564403""","""associate""","""2110""","[""PMID:17412732"", ""PMID:18836889"", … ""PMID:23727839""]","[""Chemical"", ""Disease""]","[""Gene""]"


[2026-03]

Looks like there's some MESH IDs assigned to Chemical + Disease.

It doesn't really matter:
* "associate" won't map to a predicate with sub/obj category restrictions
* NodeNorming will fix the category anyways. 

Looking at them...
* [MESH:C564403](https://id.nlm.nih.gov/mesh/C564403.html): Coenzyme Q10 Deficiency => Disease
* [MESH:C562477](https://id.nlm.nih.gov/mesh/C562477.html): Halothane Hepatitis => Disease
* [MESH:C536778](https://id.nlm.nih.gov/mesh/C536778.html): Systemic carnitine deficiency => Disease

So...try only keeping "first" value for entity type cols. 

In [87]:
merge2 = (relation_df_filtered
        .group_by(GROUP_BY_COLUMNS)
        .agg(
            pl.col("pmid").drop_nulls().unique().sort().alias("pmid_set"),
            pl.col("entity1_type").first().alias("entity1_type"),
            pl.col("entity2_type").first().alias("entity2_type"),
        ))

In [90]:
## check those same rows

merge2.join(multi_types, on=GROUP_BY_COLUMNS, how="semi")

entity1_id,relation,entity2_id,pmid_set,entity1_type,entity2_type
str,str,str,list[str],str,str
"""MESH:C562477""","""associate""","""1571""","[""PMID:10851284"", ""PMID:15456790"", … ""PMID:9877205""]","""Chemical""","""Gene"""
"""MESH:C024989""","""associate""","""MESH:C564403""","[""PMID:17442627"", ""PMID:18387363"", … ""PMID:9266503""]","""Chemical""","""Disease"""
"""MESH:C536778""","""associate""","""7124""","[""PMID:18462845"", ""PMID:41640562""]","""Chemical""","""Gene"""
"""MESH:C564403""","""associate""","""56997""","[""PMID:18319072"", ""PMID:24048965"", … ""PMID:41769026""]","""Disease""","""Gene"""
"""MESH:C564403""","""associate""","""2110""","[""PMID:17412732"", ""PMID:18836889"", … ""PMID:23727839""]","""Disease""","""Gene"""
